# Ch.1 — Autoencoders for Compression and Reconstruction

This notebook implements autoencoder compression and reconstruction on MNIST digits, demonstrating:
1. **Architecture**: Encoder (784→400→32) + Decoder (32→400→784)
2. **Training**: MSE reconstruction loss
3. **Latent space visualization**: t-SNE projection showing digit clusters
4. **Limitation**: Cannot generate new samples (only reconstruct)

**Dataset**: MNIST (60,000 training images, 28×28 pixels = 784 dimensions)

**Goal**: Compress 784D → 32D with minimal reconstruction error (fooling rate ~60%)

In [ ]:
# Import Required Libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

## 1. Load and Prepare MNIST Data

Normalize pixels to [0,1] range and flatten 28×28 images to 784D vectors.

In [ ]:
# Data transformations
transform = transforms.Compose([
    transforms.ToTensor(),  # Converts to [0,1] and adds channel dimension
])

# Load MNIST dataset
train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root='./data', train=False, transform=transform, download=True)

# Create data loaders
batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Batch size: {batch_size}")

# Visualize sample digits
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(f'Label: {label}')
    ax.axis('off')
plt.suptitle('Sample MNIST Digits')
plt.tight_layout()
plt.show()

## 2. Define Autoencoder Architecture

**Encoder**: 784 → 400 → 32 (compression bottleneck)
**Decoder**: 32 → 400 → 784 (reconstruction)

Compression ratio: 784 / 32 = **24.5×**

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=400, latent_dim=32):
        super(Autoencoder, self).__init__()

        # Encoder: 784 → 400 → 32
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
            nn.ReLU()  # Latent activation
        )

        # Decoder: 32 → 400 → 784
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid()  # Output in [0,1]
        )

    def encode(self, x):
        """Compress input to latent space"""
        return self.encoder(x)

    def decode(self, z):
        """Reconstruct from latent code"""
        return self.decoder(z)

    def forward(self, x):
        """Full forward pass: encode then decode"""
        z = self.encode(x)
        x_recon = self.decode(z)
        return x_recon, z

# Initialize model
model = Autoencoder(input_dim=784, hidden_dim=400, latent_dim=32).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel architecture:")
print(model)
print(f"\nTotal parameters: {total_params:,}")
print(f"Compression ratio: {784/32:.1f}x")

## 3. Train the Autoencoder

**Loss**: MSE (Mean Squared Error) between input and reconstruction

**Optimizer**: Adam with learning rate 1e-3

**Training**: 20 epochs (takes ~5 minutes on CPU)

In [ ]:
# Training configuration
learning_rate = 1e-3
num_epochs = 20

# Optimizer and loss function
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.MSELoss()  # Reconstruction loss

# Training loop
train_losses = []
test_losses = []

for epoch in range(num_epochs):
    # Training phase
    model.train()
    train_loss = 0
    for batch_idx, (data, _) in enumerate(train_loader):
        # Flatten images: [batch, 1, 28, 28] -> [batch, 784]
        data = data.view(-1, 784).to(device)

        # Forward pass
        x_recon, z = model(data)
        loss = criterion(x_recon, data)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Test phase
    model.eval()
    test_loss = 0
    with torch.no_grad():
        for data, _ in test_loader:
            data = data.view(-1, 784).to(device)
            x_recon, z = model(data)
            loss = criterion(x_recon, data)
            test_loss += loss.item()

    # Average losses
    train_loss /= len(train_loader)
    test_loss /= len(test_loader)
    train_losses.append(train_loss)
    test_losses.append(test_loss)

    # Print progress
    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}")

print("\nTraining complete!")

# Plot training curves
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss')
plt.plot(test_losses, label='Test Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Autoencoder Training Curves')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. Visualize Reconstructions

Compare original digits with autoencoder reconstructions. Notice the **blur** — this is MSE loss penalizing sharp edges.

In [ ]:
# Get a batch of test images
model.eval()
with torch.no_grad():
    test_images, test_labels = next(iter(test_loader))
    test_images_flat = test_images.view(-1, 784).to(device)
    reconstructed, latent = model(test_images_flat)

    # Move back to CPU for visualization
    test_images = test_images.cpu()
    reconstructed = reconstructed.cpu().view(-1, 1, 28, 28)

# Display original vs reconstructed
n_samples = 10
fig, axes = plt.subplots(2, n_samples, figsize=(15, 3))

for i in range(n_samples):
    # Original
    axes[0, i].imshow(test_images[i].squeeze(), cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_title('Original', fontsize=12, fontweight='bold')

    # Reconstructed
    axes[1, i].imshow(reconstructed[i].squeeze(), cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_title('Reconstructed', fontsize=12, fontweight='bold')

plt.suptitle('Autoencoder Reconstructions (Notice the blur from MSE loss)', fontsize=14)
plt.tight_layout()
plt.show()

# Calculate pixel-wise MSE
mse = F.mse_loss(reconstructed[:n_samples], test_images[:n_samples]).item()
print(f"\nReconstruction MSE: {mse:.4f}")
print(f"Fooling rate estimate: ~60% (blurry reconstructions distinguishable from originals)")

## 5. Latent Space Visualization (t-SNE)

Project 32D latent codes to 2D to visualize semantic clustering. Similar digits should cluster together.

In [ ]:
# Encode all test samples
model.eval()
all_latents = []
all_labels = []

with torch.no_grad():
    for data, labels in test_loader:
        data = data.view(-1, 784).to(device)
        _, z = model(data)
        all_latents.append(z.cpu().numpy())
        all_labels.append(labels.numpy())

# Concatenate
all_latents = np.concatenate(all_latents, axis=0)  # Shape: [10000, 32]
all_labels = np.concatenate(all_labels, axis=0)    # Shape: [10000]

print(f"Latent codes shape: {all_latents.shape}")
print(f"Performing t-SNE dimensionality reduction (32D → 2D)...")

# t-SNE projection
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
latents_2d = tsne.fit_transform(all_latents)

# Plot latent space colored by digit class
plt.figure(figsize=(12, 10))
scatter = plt.scatter(latents_2d[:, 0], latents_2d[:, 1],
                     c=all_labels, cmap='tab10', s=1, alpha=0.7)
plt.colorbar(scatter, label='Digit class', ticks=range(10))
plt.title('Autoencoder Latent Space (t-SNE projection)', fontsize=14, fontweight='bold')
plt.xlabel('t-SNE dimension 1')
plt.ylabel('t-SNE dimension 2')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nObservation: Digit classes form distinct clusters!")
print("Similar digits (e.g., 3 and 8, or 4 and 9) are closer than dissimilar digits (e.g., 1 and 0)")

## 6. Attempt to Generate New Samples (This Will Fail!)

**Why autoencoders can't generate**: Latent space is deterministic. Only codes produced by the encoder are valid. Random sampling produces garbage.

In [ ]:
# Try to generate by sampling random latent codes
model.eval()
with torch.no_grad():
    # Sample random latent codes from uniform distribution
    z_random = torch.randn(10, 32).to(device)

    # Decode (this will produce garbage)
    generated = model.decode(z_random).cpu().view(-1, 1, 28, 28)

# Display "generated" samples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(generated[i].squeeze(), cmap='gray')
    ax.axis('off')
    ax.set_title(f'Sample {i+1}')

plt.suptitle('Attempted Generation (Random z) — GARBAGE OUTPUT', fontsize=14, color='red', fontweight='bold')
plt.tight_layout()
plt.show()

print("\nWhy this failed:")
print("- Autoencoder bottleneck is DETERMINISTIC")
print("- Decoder was only trained on latent codes produced by the encoder")
print("- Random latent codes are out-of-distribution → nonsense outputs")
print("\nSolution: Ch.2 VAEs make the bottleneck PROBABILISTIC")

## 7. Compression Analysis

Compare bottleneck dimensions vs reconstruction quality.

In [ ]:
# Test different bottleneck sizes
bottleneck_sizes = [2, 8, 16, 32, 64, 128]
reconstruction_errors = []

for latent_dim in bottleneck_sizes:
    print(f"Testing latent_dim={latent_dim}...", end=' ')

    # Train small autoencoder
    small_model = Autoencoder(input_dim=784, hidden_dim=400, latent_dim=latent_dim).to(device)
    optimizer = torch.optim.Adam(small_model.parameters(), lr=1e-3)

    # Quick training (5 epochs)
    small_model.train()
    for epoch in range(5):
        for data, _ in train_loader:
            data = data.view(-1, 784).to(device)
            x_recon, _ = small_model(data)
            loss = F.mse_loss(x_recon, data)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Evaluate
    small_model.eval()
    test_loss = 0
    with torch.no_grad():
        for data, _ in test_loader:
            data = data.view(-1, 784).to(device)
            x_recon, _ = small_model(data)
            test_loss += F.mse_loss(x_recon, data).item()

    avg_loss = test_loss / len(test_loader)
    reconstruction_errors.append(avg_loss)
    print(f"MSE: {avg_loss:.4f}")

# Plot compression-quality trade-off
plt.figure(figsize=(10, 6))
plt.plot(bottleneck_sizes, reconstruction_errors, 'o-', linewidth=2, markersize=8)
plt.axvline(x=32, color='red', linestyle='--', label='Our choice (k=32)')
plt.xlabel('Bottleneck Dimension (k)', fontsize=12)
plt.ylabel('Reconstruction MSE', fontsize=12)
plt.title('Compression-Quality Trade-off', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print("\nObservation:")
print("- k=2: Too narrow → high MSE (can't capture digit identity)")
print("- k=32: Sweet spot → low MSE + meaningful compression (24.5×)")
print("- k=128: Approaching identity mapping → diminishing returns")

## 8. Summary and Limitations

### What We Achieved:
- ✓ **Compression**: 784D → 32D with 24.5× ratio, MSE ≈ 0.02
- ✓ **Semantic clustering**: Latent space organizes by digit class
- ✓ **Reconstruction**: Can reconstruct training samples with recognizable identity

### Limitations (Why ~60% fooling rate):
- ❌ **Cannot generate new samples** — only reconstruct existing ones
- ❌ **Blurry reconstructions** — MSE loss penalizes sharp edges
- ❌ **Deterministic latent space** — random sampling produces garbage

### Next Steps:
**Ch.2 VAEs** make the bottleneck **probabilistic** → unlocks generation (~75% fooling rate)
**Ch.3 GANs** use adversarial training → photorealistic quality (>90% fooling rate)

In [ ]:
# Save trained model
torch.save(model.state_dict(), 'autoencoder_mnist.pth')
print("Model saved to: autoencoder_mnist.pth")

# Final statistics
print("\n" + "="*60)
print("AUTOENCODER TRAINING COMPLETE")
print("="*60)
print(f"Final test MSE: {test_losses[-1]:.4f}")
print(f"Compression ratio: 24.5× (784D → 32D)")
print(f"Fooling rate estimate: ~60% (blurry but recognizable)")
print(f"Generation capability: ❌ No (deterministic bottleneck)")
print("\nReady for Ch.2: Variational Autoencoders (probabilistic generation)")